# Prompt Regression Tester

**The question this tool answers:** *"Did my last change to the prompt make the AI worse?"*

When you edit an AI app's prompt (the instructions you give the model), it's easy to fix one thing and secretly break another. A **regression** is when a change quietly breaks something that used to work. This tool catches regressions automatically:

1. **Baseline** — run a test set against the current, known-good prompt and save the scores. This is "what good looks like."
2. **Candidate** — someone edits the prompt. Run the same test set against the new prompt.
3. **Compare** — for each question, did the score drop beyond a small tolerance?
4. **Verdict** — if any question regressed → **FAIL** (don't ship). Otherwise → **PASS**.

Think of it as a spell-checker that runs before you hit send — except it checks whether your AI still behaves correctly after a prompt edit.

> This reuses the **LLM-as-judge** idea from the evaluation framework: a second model grades each answer. If you built that project, this will feel familiar — the new part is the **baseline vs. candidate comparison** and the **pass/fail gate**.

Run cells top to bottom with `Shift + Enter`.

## Step 1 — Setup

Install the library and connect to the AI, exactly as before. Your key lives in Colab Secrets (🔑 panel), named `ANTHROPIC_API_KEY`.

In [ ]:
!pip install anthropic duckdb pandas -q
print("Installed.")

In [ ]:
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
print("Client ready.")

In [ ]:
TARGET_MODEL = "claude-sonnet-4-6"   # the app whose prompt we are testing
JUDGE_MODEL  = "claude-sonnet-4-6"   # grades each answer

# How much a score may drop before we call it a regression.
# 0.10 means: a 0.10 drop on any question fails the test. Tune to taste.
REGRESSION_TOLERANCE = 0.10

print("Config set. Tolerance:", REGRESSION_TOLERANCE)

## Step 2 — The test set

These are the fixed questions we re-run every time the prompt changes. Each has a source document (`context`) and a known-correct `reference_answer`. This is a **customer-support** knowledge base — a realistic place where you'd tweak the wording of a prompt and risk breaking things.

In [ ]:
test_set = [
    {
        "id": "t001",
        "question": "How do I reset my password?",
        "context": "To reset your password, go to the login page and click 'Forgot Password'. Enter your registered email address and we will send a reset link valid for 30 minutes. If you do not receive the email, check your spam folder or contact support@example.com.",
        "reference_answer": "Click 'Forgot Password' on the login page, enter your registered email, and use the reset link sent to you (valid 30 minutes)."
    },
    {
        "id": "t002",
        "question": "What is your refund policy?",
        "context": "We offer full refunds within 30 days of purchase for unused products in their original packaging. Refunds are processed to the original payment method within 5-7 business days. Digital products are non-refundable once downloaded.",
        "reference_answer": "Full refunds within 30 days for unused products in original packaging, processed in 5-7 business days. Digital products are non-refundable once downloaded."
    },
    {
        "id": "t003",
        "question": "How long does shipping take?",
        "context": "Standard shipping takes 3-5 business days within the continental United States. Express shipping is available for an additional fee and takes 1-2 business days. International shipping takes 7-14 business days depending on the destination country.",
        "reference_answer": "Standard shipping is 3-5 business days in the continental US, express is 1-2 days, and international is 7-14 business days."
    },
    {
        "id": "t004",
        "question": "Can I change my subscription plan?",
        "context": "You can upgrade or downgrade your subscription plan at any time from your account settings. Upgrades take effect immediately and you are charged a prorated amount. Downgrades take effect at the start of your next billing cycle.",
        "reference_answer": "Yes, change plans anytime in account settings. Upgrades are immediate with prorated charges; downgrades start at the next billing cycle."
    },
    {
        "id": "t005",
        "question": "Is my payment information secure?",
        "context": "All payment information is encrypted using industry-standard TLS encryption. We do not store your full card number on our servers. Payments are processed through PCI-DSS compliant third-party providers.",
        "reference_answer": "Yes. Payment data is encrypted with TLS, full card numbers are not stored, and payments go through PCI-DSS compliant providers."
    },
    {
        "id": "t006",
        "question": "How do I contact customer support?",
        "context": "Our customer support team is available Monday through Friday, 9 AM to 6 PM Eastern Time. You can reach us by email at support@example.com or by phone at 1-800-555-0199. Live chat is available on our website during business hours.",
        "reference_answer": "Email support@example.com, call 1-800-555-0199, or use live chat on the website, Monday-Friday 9 AM-6 PM ET."
    },
    {
        "id": "t007",
        "question": "Do you offer a free trial?",
        "context": "New users can start a 14-day free trial with full access to all features. No credit card is required to begin the trial. At the end of the trial, you can choose a paid plan or your account will automatically switch to the limited free tier.",
        "reference_answer": "Yes, a 14-day free trial with full access and no credit card required. After it ends you pick a paid plan or move to the free tier."
    },
    {
        "id": "t008",
        "question": "How do I cancel my account?",
        "context": "To cancel your account, go to account settings and select 'Cancel Account'. Your account remains active until the end of your current billing period. After cancellation, your data is retained for 90 days in case you wish to reactivate.",
        "reference_answer": "Go to account settings and select 'Cancel Account'. It stays active until the billing period ends, and your data is kept for 90 days."
    }
]

print(f"Loaded {len(test_set)} test cases.")

## Step 3 — The two prompts we'll compare

This is the heart of the project. A **system prompt** is the standing instruction that shapes how the AI answers every question.

- `BASELINE_PROMPT` — the current, trusted version. Careful and grounded: answer only from the context, and say so if the answer isn't there.
- `CANDIDATE_PROMPT` — a proposed edit. Here we simulate a very common real-world change: someone makes the assistant "friendlier and more conversational." Looks harmless — but watch what it does to faithfulness.

Later you'll swap in your *own* candidate prompts to test your real edits. For the demo, this friendly-rewrite is deliberately chosen because it subtly encourages the model to embellish beyond the source — the exact kind of regression this tool is built to catch.

In [ ]:
BASELINE_PROMPT = """You are a customer support assistant. Answer the user's question using ONLY the information in the provided context. If the context does not contain the answer, say you don't have that information. Be accurate and concise."""

# A proposed edit: "make it warmer and more helpful." Seems fine...
CANDIDATE_PROMPT = """You are a super friendly, enthusiastic customer support assistant! Make the customer feel great and always sound positive and reassuring. Feel free to add helpful extra suggestions to make their day better. Answer their question warmly."""

print("Two prompts defined: BASELINE (careful) and CANDIDATE (friendly rewrite).")

## Step 4 — Get an answer *under a given prompt*

The key difference from the evaluation project: this `get_answer` takes the **system prompt** as an argument. Same question, different prompt → possibly very different answer. That's what lets us compare baseline vs. candidate fairly — same questions, only the prompt changes.

(The `system=` parameter is how you set the standing instruction for the model, separately from the user's question.)

In [ ]:
def get_answer(system_prompt, question, context):
    """Answer a question under a specific system prompt."""
    user_message = f"""Context:
{context}

Question: {question}"""

    response = client.messages.create(
        model=TARGET_MODEL,
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text.strip()

## Step 5 — The faithfulness judge

Same LLM-as-judge technique as the evaluation framework. We grade **faithfulness** — is every claim in the answer supported by the context? This is the metric most likely to break when a prompt tells the model to be chatty and "add helpful extras," because those extras often aren't in the source.

In [ ]:
import re

def judge_faithfulness(answer, context):
    """Score 0-1 how well the answer is supported by the context."""
    judge_prompt = f"""You are grading whether an answer is faithful to a source context.
Faithful means every claim in the answer is supported by the context, with nothing invented or added.

Context:
{context}

Answer:
{answer}

Score faithfulness from 0.0 (adds unsupported claims) to 1.0 (fully supported).
Return ONLY the number."""

    resp = client.messages.create(
        model=JUDGE_MODEL, max_tokens=10,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    raw = resp.content[0].text.strip()
    try:
        return float(raw)
    except ValueError:
        m = re.search(r"[0-1](?:\.\d+)?", raw)
        return float(m.group()) if m else None

## Step 6 — Score an entire prompt across the test set

This helper runs one prompt against every test case and returns a score per question. We'll call it twice — once for the baseline, once for the candidate — and then compare.

In [ ]:
def score_prompt(system_prompt, label):
    """Run one prompt over the whole test set. Returns {question_id: score}."""
    scores = {}
    print(f"Scoring [{label}] prompt...")
    for case in test_set:
        answer = get_answer(system_prompt, case["question"], case["context"])
        score = judge_faithfulness(answer, case["context"])
        scores[case["id"]] = score
        print(f"  {case['id']}: {score}")
    return scores

## Step 7 — Establish the baseline

Run the trusted prompt and record its scores. In a real workflow you'd save this baseline once and reuse it; here we compute it fresh so the notebook is self-contained.

In [ ]:
baseline_scores = score_prompt(BASELINE_PROMPT, "BASELINE")
print("\nBaseline established.")

## Step 8 — Test the candidate prompt

Now run the proposed edit over the *same* questions.

In [ ]:
candidate_scores = score_prompt(CANDIDATE_PROMPT, "CANDIDATE")
print("\nCandidate scored.")

## Step 9 — Compare and deliver a verdict 🚦

For each question we compute `baseline − candidate`. If the candidate dropped by more than `REGRESSION_TOLERANCE` on any question, that's a **regression** and the overall verdict is **FAIL** — don't ship this prompt change.

This is the gate. In a real setup you'd wire this into your code so it runs automatically on every prompt edit (that's Step 11).

In [ ]:
def compare(baseline, candidate, tolerance):
    regressions = []
    rows = []
    for qid in baseline:
        b = baseline[qid]
        c = candidate.get(qid)
        if b is None or c is None:
            continue
        delta = c - b                      # negative = got worse
        is_regression = delta < -tolerance
        if is_regression:
            regressions.append(qid)
        rows.append((qid, b, c, delta, "REGRESSION" if is_regression else "ok"))
    return rows, regressions

rows, regressions = compare(baseline_scores, candidate_scores, REGRESSION_TOLERANCE)

print(f"{'Question':<10}{'Baseline':<10}{'Candidate':<11}{'Change':<9}{'Status'}")
print("-" * 50)
for qid, b, c, delta, status in rows:
    print(f"{qid:<10}{b:<10.2f}{c:<11.2f}{delta:<+9.2f}{status}")

print("\n" + "=" * 50)
if regressions:
    print(f"VERDICT: FAIL — {len(regressions)} regression(s): {', '.join(regressions)}")
    print("Do NOT ship this prompt change.")
else:
    print("VERDICT: PASS — no regressions. Safe to ship.")
print("=" * 50)

## Step 10 — Save the comparison for the dashboard

We write the results to a DuckDB file so the Streamlit dashboard can display them. Same storage idea as the evaluation project.

In [ ]:
import duckdb
from datetime import datetime

con = duckdb.connect("regression_results.db")
con.execute("""
    CREATE TABLE IF NOT EXISTS comparisons (
        checked_at TIMESTAMP,
        question_id VARCHAR,
        baseline DOUBLE,
        candidate DOUBLE,
        delta DOUBLE,
        status VARCHAR
    )
""")
# keep only the latest comparison for a clean dashboard
con.execute("DELETE FROM comparisons")
now = datetime.now()
for qid, b, c, delta, status in rows:
    con.execute("INSERT INTO comparisons VALUES (?, ?, ?, ?, ?, ?)",
                [now, qid, b, c, delta, status])
con.close()
print("Saved results to regression_results.db")
print("Download it (Files panel) and drop it into your dashboard's data/ folder.")

## Step 11 — How this becomes a real safety gate (concept)

Right now you run this by hand. In a real project you'd connect it to **CI** (Continuous Integration) — automation that runs checks every time code changes. The flow:

1. A teammate edits the prompt and opens a pull request on GitHub.
2. GitHub Actions automatically runs this test.
3. If the verdict is FAIL, the change is **blocked** until fixed.

You don't need to build the CI part for your portfolio — but being able to *explain* this is what turns the project from "a script" into "a safety gate for shipping AI changes," which is the senior-sounding framing. Mention it in your README.

## ✅ Done — what you built

A prompt regression tester that:
- scores a **baseline** prompt and a **candidate** prompt over the same test set,
- flags any question where quality dropped beyond a tolerance,
- delivers a clear **PASS / FAIL** verdict,
- and saves results for a dashboard.

**To make it yours:** replace the demo `CANDIDATE_PROMPT` with real edits you're considering, and expand the test set to 20-30 cases in a domain you care about.

**Next:** download `regression_results.db` and open the Streamlit dashboard to see the verdict visually.